# Lab 09 — RBM and contrastive divergence

**Dataset:** [ylecun/mnist](https://huggingface.co/datasets/ylecun/mnist). Use binarized 0/1 images and a tiny RBM so energy, conditional sampling, and contrastive divergence remain inspectable.

**Workflow:** predict → implement → baseline → controlled Gibbs-step intervention → results table → conclusion.

## The math behind the experiment

An RBM assigns an energy to a visible vector $v$ and hidden vector $h$:

$$
E(v,h)=-a^Tv-b^Th-v^TWh.
$$

Lower energy means the configuration is more compatible with the learned model. Conditional probabilities are logistic functions, for example

$$
p(h_j=1\mid v)=\sigma\left(b_j+\sum_i v_iW_{ij}\right).
$$

Contrastive divergence approximates the difference between statistics under real data and statistics under a short Gibbs chain. That approximation is the reason the number of Gibbs steps becomes an important experimental variable.

In [ ]:
!pip -q install datasets torch pandas
from datasets import load_dataset
import numpy as np,pandas as pd,torch
tr=load_dataset('ylecun/mnist',split='train[:1000]')
X=torch.tensor(np.array([np.array(x).reshape(-1) for x in tr['image']])/255.,dtype=torch.float32)[:,:196]
X=(X>.5).float()
print(X.shape)

## 1. Prediction

Predict that learning will reduce reconstruction error relative to an untrained/random model. Predict that increasing Gibbs steps changes the sampling approximation and therefore the reconstruction statistics.

In [ ]:
class RBM:
    def __init__(self,v=196,h=32):
        self.W=torch.randn(v,h)*.01; self.a=torch.zeros(v); self.b=torch.zeros(h)
    def ph(self,v): return torch.sigmoid(v@self.W+self.b)
    def pv(self,h): return torch.sigmoid(h@self.W.T+self.a)
    def cd1(self,v):
        ph=self.ph(v); h=(ph>torch.rand_like(ph)).float(); pv=self.pv(h); vk=(pv>torch.rand_like(pv)).float(); phk=self.ph(vk)
        self.W += .05*(v.T@ph-vk.T@phk)/len(v); self.a += .05*(v-vk).mean(0); self.b += .05*(ph-phk).mean(0)
        return float(((v-vk)**2).mean())
rbm=RBM(); rows=[]
for epoch in range(15): rows.append([epoch+1,rbm.cd1(X)])
pd.DataFrame(rows,columns=['epoch','reconstruction_MSE'])

## 2. Controlled intervention: Gibbs steps

Keep the dataset and model size fixed. Change only the number of Gibbs transitions used by the negative phase. Compare both reconstruction error and runtime if you extend the cell with timing.

In [ ]:
def run_steps(k):
    r=RBM(); v=X.clone()
    for _ in range(10):
        h=r.ph(v)
        for _ in range(k):
            h=(h>torch.rand_like(h)).float(); v=r.pv(h)
    return float(((X-v)**2).mean())
results=pd.DataFrame([[k,run_steps(k)] for k in [1,2,5]],columns=['gibbs_steps','reconstruction_MSE'])
results

## Conclusion / answer key

Expected: the RBM should learn non-random reconstruction statistics, while Gibbs-step count changes the approximation/cost trade-off. Do **not** claim that reconstruction MSE proves better likelihood. A stronger research study would measure likelihood or a justified proxy.

**Research extension:** compare CD-1 and CD-5 at equal wall-clock budget and repeat for three seeds.